# 01 -- Pipeline 1: Product Category Classification

**Goal:** Given a product title, predict which of the 10 categories it belongs to.
This is a **supervised learning** task because the category labels already exist.

**Cells in order:**
1. Imports
2. Load precomputed embeddings + metadata
3. Encode labels (string -> integer)
4. Stratified 80/20 train/test split
5. Train Logistic Regression
6. Evaluate: accuracy + classification report
7. Confusion matrix + interpretation
8. Train Linear SVM
9. Side-by-side comparison + bar chart
10. Save best model to disk

**Prerequisite:** Run `00_embeddings.ipynb` first.

---

## Cell 1 -- Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import time
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import LinearSVC
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.metrics         import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.preprocessing  import LabelEncoder

warnings.filterwarnings('ignore')
print('Imports OK')

## Cell 2 -- Load precomputed embeddings + metadata

We load the two files saved by `00_embeddings.ipynb`.
Loading a `.npy` file takes under 1 second vs 2-5 minutes to re-encode from scratch.

In [ ]:
embeddings = np.load('embeddings.npy')    # (35311, 384)
meta       = pd.read_csv('metadata.csv')

assert len(embeddings) == len(meta), 'Row count mismatch!'

print(f'Embeddings : {embeddings.shape}  dtype={embeddings.dtype}')
print(f'Metadata   : {meta.shape}')
print(f'Columns    : {list(meta.columns)}')
print()
print(meta['category'].value_counts().to_string())

## Cell 3 -- Encode labels (string -> integer)

Scikit-learn classifiers need integer labels.
`LabelEncoder` maps each of the 10 category strings to a stable integer 0-9.
We keep the encoder object so we can decode predictions back to category names later.

In [ ]:
X = embeddings

le = LabelEncoder()
y  = le.fit_transform(meta['category'])

print('Label mapping:')
for i, cls in enumerate(le.classes_):
    count = int((y == i).sum())
    print(f'  {i:>2}  {cls:<32}  {count:>5} samples')

## Cell 4 -- Stratified 80/20 train/test split

**Why stratified?**
The 10 categories differ in size. A plain random split might under-sample a small
class in the test set, making per-class metrics unreliable.
`stratify=y` keeps each class proportion the same in both halves:
if class X is 5% of all data, it will be ~5% of train AND ~5% of test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(f'Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test : {X_test.shape[0]:,}  samples ({X_test.shape[0]/len(X)*100:.1f}%)')
print()

# Verify proportions
train_props = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_props  = pd.Series(y_test ).value_counts(normalize=True).sort_index()
prop_df = pd.DataFrame({
    'Class'  : le.classes_,
    'Train %': (train_props.values * 100).round(2),
    'Test %' : (test_props.values  * 100).round(2),
})
print('Stratification check (should be near equal):')
print(prop_df.to_string(index=False))

## Cell 5 -- Train Logistic Regression

**Why Logistic Regression (LR) over a neural classifier?**

| Property | Logistic Regression | Neural Classifier |
|---|---|---|
| Training time (35K samples) | ~5 seconds | Minutes-hours |
| GPU required | No | Often yes |
| Overfitting risk | Low | Moderate-High |
| Performance on dense embeddings | Excellent | Marginally better at best |

The 384-dim embeddings already encode rich semantics, so a **linear boundary**
in that space is sufficient to separate the 10 categories.
LR is the right default: fast, interpretable, and hard to overfit at this scale.

In [ ]:
print('Training Logistic Regression ...')
t0 = time.time()

lr = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    multi_class='auto',
    random_state=42,
    n_jobs=-1,
)
lr.fit(X_train, y_train)

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')
train_acc = accuracy_score(y_train, lr.predict(X_train))
print(f'Train accuracy (sanity check): {train_acc:.4f}')

## Cell 6 -- Evaluate: accuracy + classification report

**How to read the report:**
- **Precision** -- of all products predicted as X, what fraction are truly X?
- **Recall** -- of all truly-X products, what fraction did we predict correctly?
- **F1** -- harmonic mean of precision and recall; the right metric when class sizes differ.
- **Support** -- number of test samples in that class.

In [ ]:
y_pred_lr = lr.predict(X_test)
acc_lr    = accuracy_score(y_test, y_pred_lr)
f1_lr     = f1_score(y_test, y_pred_lr, average='macro')

print('=== Logistic Regression ===')
print(f'Test Accuracy : {acc_lr:.4f}  ({acc_lr*100:.2f}%)')
print(f'Macro F1      : {f1_lr:.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

## Cell 7 -- Confusion matrix

**How to read it:**
- **Rows** = true category, **Columns** = predicted category.
- **Diagonal** (top-left to bottom-right) = correct predictions. Goal: large values.
- **Off-diagonal** = mistakes. A bright cell at row i, column j means we predicted
  category j but the true label was category i.

**What to expect:**
Categories sharing vocabulary (e.g. Tablets and Computers both use 'gb', 'screen',
'processor') will have the highest off-diagonal values.
Visually distinct categories (Washing Machines vs Mobile Phones) will be near zero.

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_,
    linewidths=0.5, linecolor='white',
    ax=ax,
)
ax.set_xlabel('Predicted category', fontsize=11)
ax.set_ylabel('True category',      fontsize=11)
ax.set_title('Logistic Regression -- Confusion Matrix (test set)',
             fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig('confusion_matrix_lr.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top confused pairs
print('\n=== Most confused category pairs ===')
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
np.fill_diagonal(cm_df.values, 0)
confused = (
    cm_df.stack()
    .reset_index()
    .rename(columns={'level_0': 'True', 'level_1': 'Predicted', 0: 'Count'})
    .query('Count > 0')
    .sort_values('Count', ascending=False)
    .head(10)
)
print(confused.to_string(index=False))
print()
print('Interpretation: confused pairs share vocabulary that blurs the semantic boundary.')
print('Dense embeddings still separate them well overall, but the margin is softer')
print('between closely related pairs (e.g. Tablets <-> Computers).')

## Cell 8 -- Linear SVM

**Why try SVM?**
SVMs find a maximum-margin hyperplane between classes. For well-separated dense
embedding spaces this sometimes outperforms LR by 1-3%.

We use `LinearSVC` (fast, linear kernel) wrapped in `CalibratedClassifierCV`
to add `predict_proba` support -- required by the Streamlit app for displaying
confidence scores next to predictions.

In [ ]:
print('Training Linear SVM ...')
t0 = time.time()

svm_base = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm      = CalibratedClassifierCV(svm_base, cv=3)
svm.fit(X_train, y_train)

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')

y_pred_svm = svm.predict(X_test)
acc_svm    = accuracy_score(y_test, y_pred_svm)
f1_svm     = f1_score(y_test, y_pred_svm, average='macro')

print('\n=== Linear SVM ===')
print(f'Test Accuracy : {acc_svm:.4f}  ({acc_svm*100:.2f}%)')
print(f'Macro F1      : {f1_svm:.4f}')
print()
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

## Cell 9 -- Side-by-side comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric'             : ['Accuracy', 'Macro F1'],
    'Logistic Regression': [f'{acc_lr:.4f}', f'{f1_lr:.4f}'],
    'Linear SVM'         : [f'{acc_svm:.4f}', f'{f1_svm:.4f}'],
})
print('=== Model Comparison ===')
print(comparison.to_string(index=False))

winner = 'Linear SVM' if acc_svm > acc_lr else 'Logistic Regression'
margin = abs(acc_svm - acc_lr) * 100
print(f'\nWinner: {winner}  (margin: {margin:.2f} percentage points)')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
models = ['Logistic\nRegression', 'Linear SVM']
colors = ['#4C72B0', '#DD8452']

for ax, vals, title in zip(
    axes,
    [[acc_lr, acc_svm], [f1_lr, f1_svm]],
    ['Accuracy', 'Macro F1'],
):
    bars = ax.bar(models, vals, color=colors, width=0.5, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f'{v:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold',
        )
    ax.set_ylim(min(vals) - 0.05, 1.02)
    ax.set_title(title, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('LR vs LinearSVM on Sentence-Transformer Embeddings',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> model_comparison.png')

## Cell 10 -- Save best model to disk

The Streamlit app loads the classifier at startup without re-training.
We save both models -- the best one as `classifier.joblib` and LR as a
lightweight fallback -- plus the label encoder for decoding predictions.

In [ ]:
best_model = svm if acc_svm >= acc_lr else lr
best_name  = 'LinearSVM' if acc_svm >= acc_lr else 'LogisticRegression'

joblib.dump(best_model, 'classifier.joblib')
joblib.dump(lr,          'classifier_lr.joblib')
joblib.dump(le,          'label_encoder.joblib')

print(f'Saved best model ({best_name}) -> classifier.joblib')
print(f'Saved LR fallback             -> classifier_lr.joblib')
print(f'Saved label encoder           -> label_encoder.joblib')
print()
for fname in ['classifier.joblib', 'classifier_lr.joblib', 'label_encoder.joblib']:
    size_kb = os.path.getsize(fname) / 1e3
    print(f'  {fname:<35}  {size_kb:>8.1f} KB')

## Summary

| File created | Contents | Used by |
|---|---|---|
| `classifier.joblib` | Best model (LR or SVM) | Streamlit app |
| `classifier_lr.joblib` | Logistic Regression | Streamlit app |
| `label_encoder.joblib` | Category int <-> string mapping | Streamlit app |
| `confusion_matrix_lr.png` | Heatmap of LR predictions | Reference |
| `model_comparison.png` | LR vs SVM bar chart | Reference |

**Next step:** Open `02_clustering.ipynb`